# PBN evaluation harness

Compare **A / B / C** pipelines on one image, inspect metrics, optionally score by hand, and run a **budgeted sweep** with checkpointed manifests under `artifacts/`.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "packages"))

PROJECT_ROOT = ROOT.resolve()
CONFIG = PROJECT_ROOT / "config"
print("PROJECT_ROOT", PROJECT_ROOT)

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
import ipywidgets as W

from pbn_eval.pipeline import (
    PipelineVariant,
    PipelineParams,
    run_pipeline,
)
from pbn_eval.metrics import compute_metrics, lab_mse_fidelity
from pbn_eval.scoring import load_scoring_config, composite_auto_score, combined_score
from pbn_eval.preflight import estimate_sweep, load_presets
from pbn_eval.runs import append_jsonl, new_run_id, write_manifest
from pbn_eval.sweep import run_sweep

In [ ]:
def show_preflight(preset_name: str, rgb: np.ndarray):
    est = estimate_sweep(
        preset_name,
        CONFIG / "sweep_presets.yaml",
        rgb.shape,
        seconds_per_run_empirical=None,
    )
    print(json.dumps(est, indent=2))
    return est


def run_three_variants(rgb_u8: np.ndarray, params=None):
    params = params or PipelineParams()
    variants = [
        PipelineVariant.A_BASELINE,
        PipelineVariant.B_SALIENCY_PROTECT,
        PipelineVariant.C_BORDER_MERGE,
    ]
    results = []
    for v in variants:
        r = run_pipeline(rgb_u8, v, params)
        m = compute_metrics(rgb_u8, r)
        fid = lab_mse_fidelity(rgb_u8, r)
        cfg = load_scoring_config(CONFIG / "scoring_weights.yaml")
        auto = composite_auto_score(m, fid, cfg["weights"])
        results.append((v, r, m, fid, auto))
    return results


def plot_grid(rgb_u8: np.ndarray, bundle):
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    axes[0].imshow(rgb_u8)
    axes[0].set_title("Input")
    axes[0].axis("off")
    for ax, (v, r, m, fid, auto) in zip(axes[1:], bundle):
        ax.imshow(r.quantized_rgb)
        ax.set_title(f"{v.value}\nn={m.n_regions} tiny={m.tiny_region_fraction:.3f}")
        ax.axis("off")
    plt.tight_layout()
    plt.show()


def log_run(rgb_u8: np.ndarray, bundle, human, preset: str):
    cfg = load_scoring_config(CONFIG / "scoring_weights.yaml")
    blend = float(cfg.get("human_blend", 0.35))
    for v, r, m, fid, auto in bundle:
        rid = new_run_id()
        total = combined_score(auto, human, blend)
        rec = {
            "run_id": rid,
            "preset": preset,
            "variant": v.value,
            "params": r.params,
            "metrics": m.__dict__,
            "fidelity_mse": fid,
            "auto_scores": auto,
            "combined": total,
            "human": human,
        }
        append_jsonl(PROJECT_ROOT, rec)
        write_manifest(PROJECT_ROOT, rid, rec)
    print("Logged to", PROJECT_ROOT / "artifacts" / "runs.jsonl")

In [ ]:
# Optional: load a file from disk instead of widget (path relative to notebook cwd)
# from PIL import Image
# rgb_u8 = np.array(Image.open("../assets/sample.png").convert("RGB"))

uploader = W.FileUpload(accept="image/*", multiple=False)
display(uploader)

In [ ]:
from PIL import Image
import io


def bytes_to_rgb_u8(content: bytes) -> np.ndarray:
    im = Image.open(io.BytesIO(content)).convert("RGB")
    return np.array(im)


if not uploader.value:
    # Demo gradient if no upload yet
    rgb_u8 = np.zeros((128, 128, 3), dtype=np.uint8)
    rgb_u8[:, :64] = [220, 120, 80]
    rgb_u8[:, 64:] = [40, 90, 180]
    print("No upload; using demo patch")
else:
    key = next(iter(uploader.value))
    rgb_u8 = bytes_to_rgb_u8(uploader.value[key]["content"])
    print("Loaded", key, rgb_u8.shape)

show_preflight("fast_tight", rgb_u8)
params = PipelineParams()
bundle = run_three_variants(rgb_u8, params)
plot_grid(rgb_u8, bundle)

rows = []
for v, r, m, fid, auto in bundle:
    rows.append(
        {
            "variant": v.value,
            "n_regions": m.n_regions,
            "tiny_frac": m.tiny_region_fraction,
            "mean_dE_adj": m.mean_adjacent_delta_e,
            "auto_total": auto["auto_total"],
        }
    )
import pandas as pd

display(pd.DataFrame(rows))

HUMAN = None
# Example:
# HUMAN = {"subject_clarity": 4, "paintability": 4, "background_simplicity": 3}

log_run(rgb_u8, bundle, HUMAN, preset="fast_tight")

In [ ]:
# Budgeted sweep (checkpoint after each variant batch). Uses preset from sweep_presets.yaml.
PRESET_NAME = "fast_tight"  # try deep_dive or overnight when stepping away
presets = load_presets(CONFIG / "sweep_presets.yaml")["presets"]
show_preflight(PRESET_NAME, rgb_u8)

sweep_results = run_sweep(
    rgb_u8,
    project_root=PROJECT_ROOT,
    preset_name=PRESET_NAME,
    preset=presets[PRESET_NAME],
    scoring_config_path=CONFIG / "scoring_weights.yaml",
    seed=42,
    human_scores=HUMAN,
)
print("batches", len(sweep_results))
print("best combined", max(s["combined"] for s in sweep_results) if sweep_results else None)